# Chapter 1 — Kaggle runner

**Settings: GPU T4 ×2 · Internet ON · Persistence "Variables and Files"**

For anything over ~30 minutes use **Save & Run All (Commit)**. An interactive
session dies with your browser tab — that is what killed the C/G run at 42%.

---

### What this notebook is

A rewrite of the original, which had accumulated: two setup cells, a `-m-chapters`
typo that silently broke the WN11 analysis, `--strategy random` superseded three
cells later by `--strategy type_consistent`, condition **D trained twice**, and a
loop re-evaluating seven conditions that `--train --evaluate` had already scored.

### Rules baked in

| | |
|---|---|
| **Order** | A, B → S → C, G → D → E. Priority, not alphabetical |
| **Resume** | a condition whose adapter exists is skipped. Re-run any cell after a disconnect |
| **Pairing** | two independent processes, one per T4. **Never** DataParallel — it breaks autocast |
| **Negatives** | test negatives are **type-consistent**. That is what removed the type-tag leak (62.4% → 51.3%) |


## 0 · Setup


In [ ]:
REPO_URL = 'https://github.com/lynda-lagh/contribution-.git'
DEST     = '/kaggle/working/repo'
DATASET  = 'YAGO3-10'
LIMIT    = 2000

import os, sys, json, glob, time, socket, subprocess
from pathlib import Path

try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError:
    raise SystemExit('No network. Settings -> Internet -> ON, then re-run.')

def sh(*cmd, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if check and r.returncode:
        raise SystemExit(f"$ {' '.join(cmd)}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

# reset --hard only touches TRACKED files. data/*/built/, checkpoints/ and
# results/*.json are gitignored, so they survive.
# NEVER run `git clean -fdx` here: that WOULD delete them.
before = sh('git', '-C', DEST, 'rev-parse', '--short', 'HEAD', check=False) or '(none)'
if os.path.isdir(f'{DEST}/.git'):
    sh('git', '-C', DEST, 'fetch', '--depth', '1', 'origin', 'main')
    sh('git', '-C', DEST, 'reset', '--hard', 'FETCH_HEAD')
else:
    sh('git', 'clone', '--depth', '1', REPO_URL, DEST)

os.chdir(DEST)
sys.path.insert(0, DEST)
print('repo @', sh('git', '-C', DEST, 'log', '-1', '--pretty=%h  %ad  %s', '--date=short'))
print('moved', before, '->', sh('git', '-C', DEST, 'rev-parse', '--short', 'HEAD'))
for p in ('data', 'checkpoints', 'results'):
    n = sum(1 for _ in Path(p).rglob('*')) if Path(p).is_dir() else 0
    print(f'  kept {p+"/":14s} {n:5d} files')


In [ ]:
# ── pinned stack ────────────────────────────────────────────────────────────
# transformers 5.x breaks torchao and drops bitsandbytes; torchao 0.10 (Kaggle's)
# makes transformers refuse to import, which kills LoRA. Nothing here uses it.
PIN = '4.57.6'

def stack():
    r = subprocess.run([sys.executable, '-c',
        "import json,peft,transformers;"
        "print(json.dumps({'peft':peft.__version__,'tf':transformers.__version__}))"],
        capture_output=True, text=True)
    return json.loads(r.stdout) if r.returncode == 0 else None

s = stack()
if not (s and s['tf'] == PIN):
    print('installing…  have:', s)
    !pip install -q -r requirements.txt
    !pip install -q peft "transformers=={PIN}"
!pip uninstall -y -q torchao 2>/dev/null
!pip install -q tqdm

import torch
print('\nstack:', stack())
print('torch', torch.__version__, '| gpus', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' ', i, torch.cuda.get_device_name(i))


In [ ]:
# 22 tests, a DIFFERENT random graph each run — the seed is printed.
!python -m chapter1.test_chapter1


## 1 · Data

Every step skips if already done.

⚠️ **The test negatives are type-consistent, and that is deliberate.** With
uniformly random negatives the rule *"tail tag names the query relation → Yes"*
scored **62.4%** on its own, so every typed condition started 12 points ahead of
every untyped one before learning anything. Type-consistent negatives drop that
to **51.3%**. Do not revert without re-running `check_type_leak`.

The cost: type-consistent corruptions are plausible facts, so more collide with
the graph (690 rejected) and the survivors are likelier to be true-but-unrecorded.
That is the closed-world exposure, and it belongs in the paper.


In [ ]:
def sh_show(cmd):
    print('$', cmd, flush=True)
    return subprocess.call(cmd, shell=True)

if not Path('data/YAGO3-10/train.tsv').exists():
    sh_show('python -m scripts.fetch_data --datasets YAGO3-10 WN11')
else:
    print('[skip] raw datasets present')

probe = subprocess.run([sys.executable, '-c',
    "import sys; sys.path.insert(0,'.');"
    "from src.data.loaders import load_kg;"
    "kg=load_kg('YAGO3-10','data');"
    "print(sum(1 for t in kg.test if t.label is not None))"],
    capture_output=True, text=True)
labelled = int((probe.stdout or '0').strip() or 0)
if labelled:
    print(f'[skip] YAGO3-10 test already labelled ({labelled:,} rows)')
else:
    sh_show('python -m scripts.make_test_negatives --dataset YAGO3-10 '
            '--strategy type_consistent --seed 42')


In [ ]:
!python -m chapter1.validate --dataset YAGO3-10 WN11
!python -m chapter1.data --all --dataset YAGO3-10
!python -m chapter1.check_type_leak --dataset YAGO3-10 --condition C D E G --json results/type_leak_YAGO3-10.json


### Changing the test negatives (only if you mean to)

The cell above **skips** if `test.tsv` already carries labels, so re-running it
never silently rebuilds your test set. To actually change the strategy you have
to say so — `--regenerate` rewinds to `test.original.tsv` first, so the guard can
only ever wind back to the untouched original and can never destroy shipped
labels like WN11's.

⚠️ **This invalidates every result already computed.** The test set changes, so
all accuracies, the type-tag floor and the closed-world rejection count have to
be recomputed. Only do it deliberately.


In [ ]:
# UNCOMMENT DELIBERATELY. Changes the test set; invalidates existing results.
#
# type-consistent (current):  tag-only floor 51.3%, 690 candidates rejected
# random (the original):      tag-only floor 62.4%  <- the leak, do not go back
#
# !python -m scripts.make_test_negatives --dataset YAGO3-10 --strategy type_consistent --seed 42 --regenerate
# !python -m chapter1.validate --dataset YAGO3-10
# !python -m chapter1.data --all --dataset YAGO3-10
# !python -m chapter1.check_type_leak --dataset YAGO3-10 --condition C D E G
print('nothing run — uncomment above if you really want to rebuild the test set')


In [ ]:
# Optional: understand the graph before designing on it. Free.
!python -m chapter1.profile_data --dataset YAGO3-10 --json results/profile.json


## 2 · Train and evaluate

| | condition | isolates | why this early |
|---|---|---|---|
| 1 | **A, B** | entity names | **the decomposition.** Nothing else means anything without it |
| 2 | **S** | name↔entity binding | the one control that can invalidate the headline |
| 3 | **C, G** | type information | read C against the **51.3% floor**, not against B |
| 4 | **D** | negative hardness | |
| 5 | **E** | negative count | 70k instances, ~2 h — pair it with nothing |

⚠️ A, B, C, G and S train on **1 random negative** but are tested on
**type-consistent** ones. D and E train on hard negatives, so they are matched to
the test set when the others are not. A D-over-C gain is partly that alignment,
not hardness. Report D and E against **C**, not against A.


In [ ]:
def adapter_of(cond, ds=None):
    return Path('checkpoints') / f'ch1-{ds or DATASET}-{cond}'

def trained(cond, ds=None):
    return (adapter_of(cond, ds) / 'adapter_config.json').exists()

def pair(a, b):
    """Two INDEPENDENT processes, one per T4. Never DataParallel: two visible
    devices make HF Trainer wrap the model and autocast never reaches the
    replicas -> 'mat1 and mat2 must have the same dtype'."""
    pa = subprocess.Popen(f'CUDA_VISIBLE_DEVICES=0 {a}', shell=True)
    pb = subprocess.Popen(f'CUDA_VISIBLE_DEVICES=1 {b}', shell=True)
    return pa.wait(), pb.wait()

def cmd_for(cond, ds=None, train=True, evaluate=True, extra=''):
    ds = ds or DATASET
    do_train = train and not trained(cond, ds)
    if train and not do_train:
        print(f'[skip] {cond}: adapter already on disk')
    flags = ('--train ' if do_train else '') + ('--evaluate ' if evaluate else '')
    if not flags.strip():
        return None
    return (f'python -m chapter1.run --dataset {ds} --condition {cond} '
            f'{flags} --limit {LIMIT} {extra}')

def go_pair(c1, c2, **kw):
    a, b = cmd_for(c1, **kw), cmd_for(c2, **kw)
    if a and b:
        t0 = time.time(); rc = pair(a, b)
        print(f'[{c1}|{c2}] rc={rc}  {(time.time()-t0)/60:.1f} min')
    elif a or b:
        one = a or b
        t0 = time.time(); rc = subprocess.call(f'CUDA_VISIBLE_DEVICES=0 {one}', shell=True)
        print(f'rc={rc}  {(time.time()-t0)/60:.1f} min')

def go_one(cond, gpu=0, **kw):
    cmd = cmd_for(cond, **kw)
    if cmd:
        t0 = time.time()
        rc = subprocess.call(f'CUDA_VISIBLE_DEVICES={gpu} {cmd}', shell=True)
        print(f'[{cond}] rc={rc}  {(time.time()-t0)/60:.1f} min')

print(f"{'cond':6s} {'trained':>9s}")
for x in ['A','B','S','C','G','D','E']:
    print(f'{x:6s} {str(trained(x)):>9s}')


In [ ]:
go_pair('A', 'B')      # ~40 min — the decomposition


In [ ]:
go_pair('S', 'C')      # ★ S defends the chapter · C tests types


In [ ]:
go_pair('G', 'D')      # types-with-names · negative hardness


In [ ]:
# E is 70,000 instances (3.5x) — ~2 h. Nothing left to pair it with, so GPU1 is
# idle. If you want it busy, put WN11's shuffled control there:
#   !python -m chapter1.data --condition S --dataset WN11
#   pair('python -m chapter1.run --dataset YAGO3-10 --condition E --train --evaluate',
#        'python -m chapter1.run --dataset WN11 --condition S --train --evaluate')
go_one('E')


### Re-evaluate without retraining

Training and evaluation happen in one call, so normally you never need this.
You need it when **training succeeded but scoring failed** — which has happened
twice: once on a missing `data/*-anon/built/` path, once on `float()` applied to
a dict inside the calibration helper. Both times the adapter was fine and the
numbers were lost.

This finds every trained condition with no result file and scores only those.
~4 min per condition. It never retrains.


In [ ]:
import glob

def has_result(cond, ds=None):
    ds = ds or DATASET
    return bool(glob.glob(f'results/*{ds}-{cond}-eval*.json'))

missing = [c for c in ['A','B','S','C','G','D','E']
           if trained(c) and not has_result(c)]

print('trained but unscored:', missing or 'none — nothing to do')
for c in missing:
    cmd = (f'python -m chapter1.run --dataset {DATASET} --condition {c} '
           f'--evaluate --limit {LIMIT}')
    print('$', cmd, flush=True)
    subprocess.call(f'CUDA_VISIBLE_DEVICES=0 {cmd}', shell=True)

# force a re-score of specific conditions (e.g. after changing --limit):
#   for c in ['A','B']:
#       subprocess.call(f'CUDA_VISIBLE_DEVICES=0 python -m chapter1.run --dataset {DATASET} --condition {c} --evaluate --limit {LIMIT}', shell=True)


## 3 · Collect — read, do not recompute

`--train --evaluate` already produced every number. The old notebook re-evaluated
all seven conditions here, serially on one GPU, for nothing.


In [ ]:
rows = []
for f in sorted(glob.glob('results/*eval*.json')):
    d = json.load(open(f))
    rows.append((Path(f).stem, d.get('acc_real'), d.get('acc_anon'), d.get('gap')))
fmt = lambda v: f'{v:.4f}' if isinstance(v, float) else str(v)
print(f"{'run':44s} {'real':>8s} {'anon':>8s} {'gap':>8s}")
for r in rows:
    print(f'{r[0]:44s} {fmt(r[1]):>8s} {fmt(r[2]):>8s} {fmt(r[3]):>8s}')
print(f'\n{len(rows)} runs found — a missing condition means its evaluation failed.')


In [ ]:
# ★ THE HEADLINE — matched arms, each adapter on the data it was trained for.
#   NOT the acc_real/acc_anon pair inside one run: that mixes in distribution
#   shift, because A never saw an `entityN` during training.
def cell(cond, key, ds=None):
    for f in glob.glob(f'results/*{ds or DATASET}-{cond}-eval*.json'):
        return json.load(open(f)).get(key)

real, anon = cell('A', 'acc_real'), cell('B', 'acc_anon')
if real and anon:
    above, mem = real - 0.5, real - anon
    print(f'  tuned  (A on real)   {real:.4f}')
    print(f'  anon   (B on anon)   {anon:.4f}')
    print(f'  above chance         {above:.4f}')
    print(f'  memorisation         {mem:.4f}')
    print(f'  residual knowledge   {above - mem:.4f}')
    print(f'  MEMORISATION SHARE   {mem/above:.1%}')
else:
    print('A and/or B not evaluated yet')

# typed ladder: G and C are the matched typed pair; compare C to the 0.513 floor
g, cc = cell('G', 'acc_real'), cell('C', 'acc_anon')
if g and cc:
    print(f'\n  typed:  G {g:.4f}   C {cc:.4f}   gap {g-cc:+.4f}')
    print(f'  C vs tag-only floor 0.513: {cc-0.513:+.4f}')
    if real:
        print(f'  G vs A: {g-real:+.4f}   (negative = types HURT when names exist)')


## 4 · Analysis · report · ranking · SMI


In [ ]:
!python -m chapter1.analysis --dataset {DATASET}
!python -m chapter1.report   --dataset {DATASET}


In [ ]:
# 50-way filtered link prediction -> Hits@K and MRR, no retraining.
for x in ['A', 'B', 'C', 'S']:
    if not trained(x):
        print(f'[skip] {x}: not trained'); continue
    cmd = (f'python -m chapter1.rank --adapter checkpoints/ch1-{DATASET}-{x} '
           f'--dataset {DATASET} --condition {x} --limit 500')
    print('$', cmd, flush=True)
    subprocess.call(f'CUDA_VISIBLE_DEVICES=0 {cmd}', shell=True)


In [ ]:
# SMI — FLAME's instrument. Slow; only A and B carry the comparison.
go_pair('A', 'B', train=False, extra='--smi')


## 5 · WN11 — the second dataset

Complete: untuned **0.6920** · tuned **0.9265** · anonymised **0.5325** →
memorisation **0.3940**, residual **0.0325**, **92.4%**. Balanced familiarity gap
**+0.0036** — flat, which localises the memorisation to *pretraining* rather than
to the fine-tuning sample.

Re-run only if those adapters were lost.


In [ ]:
# ── WN11 diagnostics: the same free checks YAGO3-10 gets ───────────────────
# ⚠️ TYPE_TAG_FLOOR['WN11'] is still None, so floor_for() falls back to 0.5 for
#    WN11's typed conditions. On YAGO3-10 that assumption was wrong by 12.4
#    points. Measure it before comparing typed results across the two datasets.
!python -m chapter1.validate --dataset WN11
!python -m chapter1.profile_data --dataset WN11 --json results/profile_WN11.json
!python -m chapter1.data --all --dataset WN11
!python -m chapter1.check_type_leak --dataset WN11 --condition C D E G --json results/type_leak_WN11.json


In [ ]:
!python -m chapter1.data --condition A B S --dataset WN11
go_pair('A', 'B', ds='WN11')
# ⚠️ the original notebook had `-m-chapters` here — a typo that made this a
#    no-op and left the WN11 result file without its tuned arm.
!CUDA_VISIBLE_DEVICES=0 python -m chapters.ch1_diagnostic.analyse --dataset WN11
!python -m chapter1.seen_unseen --dataset WN11


### Export the fine-tuned adapters

LoRA trained **1,089,536 of 1,544,803,840** parameters — 0.07%. So the
fine-tuned artefact is ~**4 MB**, not 3 GB: the base model is untouched and is
re-downloaded on load.

The folders are big for two reasons that do not need keeping:

| | why it is there | keep? |
|---|---|---|
| `checkpoint-250/`, `checkpoint-500/` | optimiser + scheduler + RNG state, written every 250 steps | ❌ dead once the best model is saved |
| `tokenizer.json`, `vocab.json` | ~10 MB, **byte-identical** in every adapter | ❌ the base model id is recorded instead |
| `adapter_model.safetensors` | **the fine-tuning** | ✅ |

`--prune` deletes the first; the export drops the second. About **30 MB → 4 MB**
per adapter. `MANIFEST.json` records base model, LoRA hyper-parameters, runtime,
peak VRAM, the fit verdict and the accuracies. `USAGE.md` has the six lines that
load one.


In [ ]:
# small, portable, reusable. --prune reclaims the optimiser state too.
!python -m scripts.export_adapters --zip --prune

!echo && echo '--- what to download ---'
!ls -la /kaggle/working/*.zip export/adapters.zip 2>/dev/null
print('\n★ export/adapters.zip is the one to keep: every fine-tuned adapter,')
print('  a few MB, reloadable without retraining. See export/adapters/USAGE.md.')


## 6 · Backup

⚠️ `/kaggle/working` does not survive an interactive session ending. **Download
this zip and the `checkpoints/` folder before you close the tab**, or commit the
notebook so Kaggle stores the output.


In [ ]:
import datetime
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
out = f'/kaggle/working/chapter1_{stamp}.zip'
!zip -qr "{out}" results/ data/*/built/manifest.json
print('wrote', out)
!du -sh checkpoints results 2>/dev/null
print('\n★ Download BOTH this zip and checkpoints/ — the adapters are ~2.7 GPU-h of work.')
